# Function Calling

In the previous lesson we built a RAG pipeline with `RAGBase.rag()` and saw it fail on typos and ambiguous queries. The pipeline is fixed: search → build prompt → LLM. The LLM is a passenger — it never sees the search results until after the prompt is built, so it can't recover from a bad query.

**An agent puts the LLM in charge.** Instead of running search ourselves, we give the LLM a search *tool*. It decides when to call it and what to search for. If the results are bad, it can try again with a different query — on its own, with no special-case code from us.

The mechanism that makes this possible is **function calling**.

## Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

In [2]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="db/faq.db"
)

index.count()

158

### Search function

We define a top-level `search` function that queries the SQLite index directly. The model will reference it by this name, so we keep the Python function name and the tool name aligned — it makes the dispatch step straightforward later.

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [4]:
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

deepseek_client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

## Asking Without Tools

First, let's see what the LLM does without any tools. We send a course-specific question and inspect the answer. The model has no access to our FAQ, so it can only respond from general knowledge — the answer will be vague and unhelpful. This is exactly why we need tools.

In [5]:
# Asking without tools - the model answers from general knowledge
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = deepseek_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
)

print(response.choices[0].message.content)

I'd be happy to help, but I don't have information about which specific course you're referring to. If you can share the course name or link, I can try to help you find enrollment details. Alternatively, you may want to check the course platform or contact the course provider directly for information on joining. Let me know how I can assist further!


## Defining the Tool and Calling with It

We describe the `search` function to the model as a JSON schema. The model doesn't see our Python code — it only sees this description. LLMs are language-agnostic; under the hood we're making HTTP calls, so the tool is described in JSON rather than Python. The same schema would work from TypeScript or any other language.

The `description` field is the most important part: **the model reads it to decide when to call the function.** The `parameters` block is a JSON Schema for the arguments, and marking `query` as `required` ensures the model always fills it in.

We then resend the same question, but this time include the tool in the request. Instead of a text answer, the response now contains a `tool_calls` entry — the model decided it needs to search the FAQ before it can answer. Notice it doesn't pass the user's question verbatim; it rewrites it into better search keywords.

In [6]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

# Now the model decides to call search instead of answering directly
response = deepseek_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[search_tool],
)

# Should show a tool_calls entry, not a text answer
print(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_00_slKIodydPUBkBMyI8lHW0346', function=Function(arguments='{"query": "Can I join the course late"}', name='search'), type='function', index=0)]


## Executing the Function and Sending the Result Back

The model can't run our Python code — it only asked us to. We parse the JSON arguments it provided, call `search`, and serialize the result.

Then we send everything back in a second API call. Two things get appended to the conversation history:

1. **The model's tool-call message** — the model needs to see its own function call in the history (LLMs are stateless; the memory is the list you send as input).
2. **The tool result** — linked to the specific call by `tool_call_id`. If the model had made multiple function calls in one turn, each result would be matched by its own ID.

In [7]:
import json

tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

# Add the model's function-call message to history
messages.append(response.choices[0].message)

# Add the tool result linked by tool_call_id
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json,
})

In [8]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s)

In [9]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_slKIodydPUBkBMyI8lHW0346', function=Function(arguments='{"query": "Can I join the course late"}', name='search'), type='function', index=0)], reasoning_content='The user is asking about joining the course. Let me search the FAQ for information about joining the course or enrollment.'),
 {'role': 'tool',
  'tool_call_id': 'call_00_slKIodydPUBkBMyI8lHW0346',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "74eb249bbf",\n    "co

## Asking the Model Again

We call the API a second time with the full expanded history: the original question, the model's decision to call `search`, and the FAQ results it got back. Now it can produce a proper, course-specific answer.

We have to replay the whole history because LLMs are stateless between API calls. If we sent only the tool result, the model would have no idea what's going on.

In [10]:
# Second call: model now has the original question + its tool call + the FAQ results
response = deepseek_client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=[search_tool],
)

print(response.choices[0].message.content)

Yes, you can still join! 🎉

According to the course FAQ, you are welcome to join at any time. However, there's an important note if you're interested in earning a **certificate**:

- **Yes, you can join** – just start learning!
- **But** if you want to receive a certificate, you need to **submit your project while submissions are still being accepted** (i.e., while the current "live" cohort is still running). Certificates are only awarded to those who complete the course with a live cohort, including submitting their project and peer-reviewing others' capstone projects.

If you're not worried about the certificate, you can follow the course in **self-paced mode** at your own convenience. No registration confirmation is needed — you can just start learning and doing the homework right away!

Would you like to know more about how the course works or what's covered?


## Token Usage and Cost

We just made two API calls instead of one. Each call costs money, so it's worth checking how much one tool-using turn actually costs.

The `usage` field on the response gives us token counts. We multiply by the per-million-token price to get a dollar figure. Note this is only for the *second* call — the first call (where the model decided to invoke `search`) also has its own cost. Two calls means we pay twice, and the second call is more expensive because we resend the full history as input.

With a real agent loop the model can make many calls in a single turn, so costs add up quickly. Keep an eye on usage while you develop.

In [11]:
usage = response.usage
print(usage.prompt_tokens, usage.completion_tokens)

def calculate_deepseek_flash_price(input_tokens, output_tokens):
    # DeepSeek-V3 pricing: $0.07/M input, $1.10/M output (check https://platform.deepseek.com/pricing)
    INPUT_PRICE_PER_MILLION = 0.07
    OUTPUT_PRICE_PER_MILLION = 1.10

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_deepseek_flash_price(usage.prompt_tokens, usage.completion_tokens)
print("Total cost: $", round(result["total_cost"], 8))

971 206
Total cost: $ 0.00029457
